## 9.2

In [ ]:
import numpy as np

def givens_hessenberg(A):
    """
    Reduce a real n×n matrix A to upper Hessenberg form H using Givens rotations.
    Returns H and Q such that H = Q.T @ A @ Q.
    """
    A = A.astype(float).copy()
    n = A.shape[0]
    Q = np.eye(n)

    # Work column by column
    for j in range(n - 2):
        # Go up the column from bottom to row j+2
        for i in range(n - 1, j + 1, -1):
            a = A[i - 1, j]
            b = A[i, j]

            if abs(b) < 1e-14:
                continue  # already zero, nothing to do

            r = np.hypot(a, b)  # sqrt(a^2 + b^2) safely
            c = a / r
            s = b / r

            # Build Givens matrix G acting on rows/cols (i-1, i)
            G = np.eye(n)
            G[i - 1, i - 1] = c
            G[i - 1, i]     = s
            G[i, i - 1] = -s
            G[i, i]     = c

            # Similarity transformation: A <- G A G^T
            A = G @ A @ G.T
            # Accumulate Q so that finally H = Q^T A_original Q
            Q = Q @ G.T

    return A, Q


# small test
if __name__ == "__main__":
    A_test = np.random.randn(5, 5)
    H, Q = givens_hessenberg(A_test)
    print("H is Hessenberg? ->", np.allclose(H[np.tril_indices(5, -2)], 0))
    print("Similarity check Q^T A Q = H? ->", np.allclose(Q.T @ A_test @ Q, H))


H is Hessenberg? -> True
Similarity check Q^T A Q = H? -> True


### b

Assume $A = A^T$ and $H = Q^T A Q$ is obtained by the procedure in (a).  
Then  

$H^T = (Q^T A Q)^T = Q^T A^T Q = Q^T A Q = H,$

so $H$ is symmetric.

From part (a) we know that $H$ is upper Hessenberg, i.e.

$h_{ij} = 0 \quad \text{for all } i > j + 1.$

By symmetry $h_{ij} = h_{ji}$. Hence whenever $i > j + 1$ we also have $j < i - 1$, so

$h_{ji} = 0 \;\Rightarrow\; h_{ij} = 0$

for all $j > i + 1$.

Thus all entries with $|i-j| > 1$ vanish: only the main diagonal and the first sub-/super-diagonals may be nonzero.  
This is exactly the definition of a **tridiagonal** matrix.




### c1

In [ ]:
import numpy as np

# Matrix from part (c1)
A = np.array([[1, 0, 1],
              [1, 1, 1],
              [2, 3, 1]], dtype=float)

n = A.shape[0]
Q = np.eye(n)

# We only need to zero A[2,0] (the (3,1) entry)
a = A[1, 0]  # a_21
b = A[2, 0]  # a_31

r = np.hypot(a, b)
c = a / r
s = b / r

# Givens rotation acting on indices 1 and 2 (rows/cols 2 and 3)
G = np.eye(n)
G[1, 1] = c
G[1, 2] = s
G[2, 1] = -s
G[2, 2] = c

H = G @ A @ G.T   # similarity
Q = Q @ G.T

print("Hessenberg form H (c1):")
print(H)
print("Check Q^T A Q = H? ->", np.allclose(Q.T @ A @ Q, H))


Hessenberg form H (c1):
[[ 1.          0.89442719  0.4472136 ]
 [ 2.23606798  2.6        -2.2       ]
 [ 0.         -0.2        -0.6       ]]
Check Q^T A Q = H? -> True


### c2

In [ ]:
import numpy as np

# Symmetric 4x4 matrix from part (c2)
A = np.array([[1, 2, 3, 1],
              [2, 1, 3, 1],
              [3, 3, 1, 0],
              [1, 3, 0, 0]], dtype=float)

n = A.shape[0]
Q = np.eye(n)
H = A.copy()

# Householder tridiagonalization
for k in range(n - 2):
    # Vector of entries below the diagonal in column k
    x = H[k+1:, k]

    # If already zero below first subdiagonal, skip
    if np.allclose(x[1:], 0):
        continue

    e1 = np.zeros_like(x)
    e1[0] = 1.0

    alpha = np.linalg.norm(x)
    # Use sign trick for numerical stability
    v = x + np.sign(x[0] if x[0] != 0 else 1.0) * alpha * e1
    v = v / np.linalg.norm(v)

    # Build Householder reflector acting on rows/cols k+1..n-1
    Hk = np.eye(n)
    Hk[k+1:, k+1:] -= 2.0 * np.outer(v, v)

    H = Hk @ H @ Hk
    Q = Q @ Hk

print("Tridiagonal (Hessenberg) form H (c2):")
print(H)
print("Check symmetry? ->", np.allclose(H, H.T))
print("Check Q^T A Q = H? ->", np.allclose(Q.T @ A @ Q, H))


Tridiagonal (Hessenberg) form H (c2):
[[ 1.00000000e+00 -3.74165739e+00  3.49788786e-16  3.01217830e-17]
 [-3.74165739e+00  4.07142857e+00 -1.43747227e+00  1.19522861e+00]
 [ 3.49788786e-16 -1.43747227e+00 -6.71428571e-01  6.41426981e-01]
 [ 3.01217830e-17 -9.32192820e-17  2.24499443e+00 -1.40000000e+00]]
Check symmetry? -> False
Check Q^T A Q = H? -> True
